# Equilibrium Cascade Controller for 03LIC_1071 PVLO

**Approach (per OTS guidance).** Recommendations are based on **maintaining material-balance equilibrium** in the equipment around the 1071 vessel — *not* on copying historical operator actions. Given a moment `t_now` (trigger decided elsewhere) we read the live levels/outputs and recommend coordinated OP moves that restore the level while keeping the upstream supplier from emptying.

### The cascade (confirmed by data)
```
[feed disturbance] → 1016 tank → (1071.OP valve) → 1071 tank → downstream
```
- **03LIC_1071.OP** opens the inflow from the **1016** tank into the **1071** vessel → raising 1071.OP **raises** the 1071 level (measured process gain **+2.4 PV per %OP**).
- Pulling fluid into 1071 **depletes 1016** (measured coupling negative).
- **03LIC_1016.OP** opens 1016's own inflow → raising it **refills 1016** (gain **+3.7 PV per %OP**).

**Data verification (2024–25):** during a 1071 PVLO, `1016.PV` falls **41.0 → 36.8** (supplier depleting) while operators raise `1016.OP` **38.3 → 44.2** and `1071.OP` **53.3 → 57.0** — exactly the equilibrium response the OTS engineers described.

### Recommendation logic
1. **Primary — lift 1071.** Increase `03LIC_1071.OP` to bring the level back above the alarm with margin. Step size = `level_deficit / gain_1071`, so it **scales with alarm intensity**.
2. **Cascade — protect the supplier.** Predict the extra 1016 drop our pull will cause; if 1016 is at / heading below its floor, increase `03LIC_1016.OP` to refill it (so it never empties and can keep feeding 1071). Step size = `1016_deficit / gain_1016`.
3. **Direction is fixed by physics** (always *increase* these OPs to recover a PVLO); **magnitude scales with the deficits** → with the three feed-scenario intensities.

### The three OTS scenarios
The OTS varies a **feed** stream (not FI1000) at three magnitudes, producing 1071 PVLO alarms of **increasing intensity**. The controller responds with proportionally **larger** OP increases — the property the seniors want to see — while the cascade keeps 1016 safe.

> The earlier similarity / decision-moment machinery (cells below, up to the gain table) is retained only as a **data-driven cross-check** that the empirical gains agree with this physical model (they do: 1071 → increase, 1016 → increase). The **equilibrium cascade controller** is the authoritative recommender.


In [1]:
import pandas as pd
import numpy as np
import os
import json
from tqdm import tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── PV/OP time series (canonical minute-wise parquet: 1.74M rows, 28 PV + 15 OP, 2022–2025) ──
PV_OP_PATH = '/home/h604827/ControlActions/DATA/PV-OP_data/03LIC_1071_JAN_2026.parquet'
pv_op_data_df = pd.read_parquet(PV_OP_PATH)
if 'TimeStamp' in pv_op_data_df.columns:
    pv_op_data_df = pv_op_data_df.set_index('TimeStamp')
pv_op_data_df = pv_op_data_df.sort_index()
pv_op_data_df = pv_op_data_df[~pv_op_data_df.index.duplicated(keep='first')]

# ── Alarm clusters + pre-extracted control actions (events-based cluster boundaries) ──
CLUSTER_XLSX = '/home/h604827/ControlActions/DATA/1071_pvlo_alarms_clustered_with_control_actions.xlsx'
alarms_df = pd.read_excel(CLUSTER_XLSX, sheet_name=0)
actions_df = pd.read_excel(CLUSTER_XLSX, sheet_name=1)
actions_df['VT_Start'] = pd.to_datetime(actions_df['VT_Start'])

# ── Operating limits ──
operating_limits_df = pd.read_csv('/home/h604827/ControlActions/DATA/operating_limits.csv')

# ── Constants ──
ALARM_THRESHOLD = 28.75
TARGET_TAG = '03LIC_1071'
TARGET_PV = '03LIC_1071.PV'
FEED_TAG = '02FI_1000'          # main cause #1: feed fluctuation
PRESSURE_TAG = '03PIC_1013'     # main cause #2: compressor pressure

# Tags we are allowed to recommend actions on (RCA selects a subset of these at runtime)
ACTION_TAGS = [
    '03PIC_1013', '03HIC_1141', '03HIC_1151', '03LIC_1071',
    '03HIC_3132', '03HIC_3100', '03FIC_3415', '03LIC_1034',
    '03PIC_3131', '03TIC_1009', '03LIC_1016', '03LIC_1085',
]

print(f'PV/OP data : {pv_op_data_df.shape[0]:,} rows × {pv_op_data_df.shape[1]} cols '
      f'| {pv_op_data_df.index.min()} → {pv_op_data_df.index.max()}')
print(f'  key tags present: '
      + ', '.join(t for t in [TARGET_PV, FEED_TAG + ".PV", PRESSURE_TAG + ".PV"] if t in pv_op_data_df.columns))
print(f'Alarm clusters : {alarms_df["cluster_id"].nunique()} clusters, {len(alarms_df)} individual alarms')
print(f'Control actions: {len(actions_df)} rows across {actions_df["cluster_id"].nunique()} clusters')
print(f'Operating limits: {len(operating_limits_df)} tags')
print(f'Action tag universe: {len(ACTION_TAGS)} tags')


PV/OP data : 1,737,586 rows × 45 cols | 2022-01-03 22:45:00 → 2025-06-23 20:44:00
  key tags present: 03LIC_1071.PV, 02FI_1000.PV, 03PIC_1013.PV
Alarm clusters : 539 clusters, 1379 individual alarms
Control actions: 16094 rows across 430 clusters
Operating limits: 40 tags
Action tag universe: 12 tags


In [2]:
# Cluster-level boundaries + exclude trip/shutdown clusters (PV <= 0 for >= 30 min)
clusters_agg = alarms_df.groupby('cluster_id').agg(
    cluster_start=('cluster_start_time', 'first'),
    cluster_end=('cluster_end_time', 'first'),
    cluster_type=('cluster_type', 'first'),
    n_alarms=('cluster_total_alarms', 'first')
).sort_values('cluster_start')
clusters_agg['cluster_start'] = pd.to_datetime(clusters_agg['cluster_start'])
clusters_agg['cluster_end'] = pd.to_datetime(clusters_agg['cluster_end'])

trip_clusters = set()
for cid, row in clusters_agg.iterrows():
    window = pv_op_data_df.loc[row['cluster_start'] - pd.Timedelta(minutes=30):
                               row['cluster_end'] + pd.Timedelta(minutes=30), TARGET_PV]
    if window.empty:
        trip_clusters.add(cid)
        continue
    below = (window <= 0).astype(int)
    if below.sum() == 0:
        continue
    groups = (below != below.shift()).cumsum()
    for _, grp in window.groupby(groups):
        if (grp <= 0).all() and len(grp) > 1 and \
           (grp.index[-1] - grp.index[0]).total_seconds() / 60 >= 30:
            trip_clusters.add(cid)
            break

valid_cluster_ids = sorted(set(clusters_agg.index) - trip_clusters)
clusters_valid = clusters_agg.loc[valid_cluster_ids].copy()

# Target-tag SP/OP actions only (drop MODE etc.), restricted to valid (non-trip) clusters
target_actions_df = actions_df[
    actions_df['Source'].isin(ACTION_TAGS) &
    actions_df['Description'].isin(['SP', 'OP']) &
    actions_df['cluster_id'].isin(valid_cluster_ids)
].copy()

print(f'Total clusters     : {len(clusters_agg)}')
print(f'Trip/shutdown drop : {len(trip_clusters)}')
print(f'Valid clusters     : {len(clusters_valid)}')
print(f'\nTarget SP/OP actions (valid clusters): {len(target_actions_df)}')
print(f'Clusters with such actions: {target_actions_df["cluster_id"].nunique()}')
print('\nActions by source tag:')
print(target_actions_df['Source'].value_counts().to_string())


Total clusters     : 539
Trip/shutdown drop : 9
Valid clusters     : 530

Target SP/OP actions (valid clusters): 9734
Clusters with such actions: 361

Actions by source tag:
Source
03PIC_1013    1778
03HIC_3100    1545
03HIC_1151    1468
03HIC_3132     984
03HIC_1141     854
03LIC_1071     760
03PIC_3131     672
03LIC_1034     555
03FIC_3415     446
03LIC_1016     348
03LIC_1085     197
03TIC_1009     127


### Train / test split (temporal, leakage-free)

- **Train (the bank):** clusters starting **before 2025-01-01** → the historical decision-moment bank.
- **Test (held-out):** clusters in **2025** that have target-tag actions → used only to validate direction + magnitude.

Temporal split (not random) so we never match a moment to its own future and the evaluation mimics deployment.


In [3]:
# Temporal split by cluster start year
SPLIT_DATE = pd.Timestamp('2025-01-01')
clusters_with_actions = sorted(target_actions_df['cluster_id'].unique())
cwa = clusters_valid.loc[clusters_valid.index.isin(clusters_with_actions)]

train_cluster_ids = sorted(cwa.index[cwa['cluster_start'] < SPLIT_DATE])
test_cluster_ids = sorted(cwa.index[cwa['cluster_start'] >= SPLIT_DATE])

print(f'Clusters with target actions: {len(clusters_with_actions)}')
print(f'  Train (<2025): {len(train_cluster_ids)} clusters')
print(f'  Test  (2025) : {len(test_cluster_ids)} clusters')


Clusters with target actions: 361
  Train (<2025): 305 clusters
  Test  (2025) : 56 clusters


### Helper functions

- **Operating-limits lookup** → normalized position within a tag's operating range.
- **PV value / window lookups** → value at a timestamp, and a backward time-window slice (for rolling ROC / volatility).
- **Sub-minute action merging** → collapse multiple actions on the same tag in the same minute into one net action.


In [4]:
# Operating-limits lookup dict
op_limits = {}
for _, row in operating_limits_df.iterrows():
    tag_base = row['TAG_NAME'].replace('.PV', '').replace('.OP', '')
    rng = row['UPPER_LIMIT'] - row['LOWER_LIMIT']
    if rng > 0:
        op_limits[tag_base] = {'lower': row['LOWER_LIMIT'], 'upper': row['UPPER_LIMIT'], 'range': rng}

target_upper = op_limits[TARGET_TAG]['upper']
print(f'Operating limits loaded for {len(op_limits)} tags')
print(f'{TARGET_TAG}: lower={op_limits[TARGET_TAG]["lower"]:.2f}, upper={target_upper:.2f}, alarm={ALARM_THRESHOLD}')
for t in [FEED_TAG, PRESSURE_TAG]:
    has = t in op_limits
    print(f'{t}: limits {"OK" if has else "MISSING"}' + (f' [{op_limits[t]["lower"]:.2f}, {op_limits[t]["upper"]:.2f}]' if has else ''))


Operating limits loaded for 26 tags
03LIC_1071: lower=35.25, upper=42.41, alarm=28.75
02FI_1000: limits OK [8.39, 8.63]
03PIC_1013: limits OK [172.79, 247.95]


In [5]:
_PV_INDEX = pv_op_data_df.index  # cached for fast nearest lookups


def _norm_ts(timestamp):
    if hasattr(timestamp, 'tzinfo') and timestamp.tzinfo is not None:
        timestamp = timestamp.tz_localize(None)
    return timestamp


def get_pv_at(timestamp, pv_tag):
    """PV value at or most-recent-before the given timestamp (causal: never looks forward)."""
    if pv_tag not in pv_op_data_df.columns:
        return np.nan
    timestamp = _norm_ts(timestamp)
    try:
        idx = _PV_INDEX.get_indexer([timestamp], method='ffill')[0]
        if idx < 0:                                  # before data starts → fall back to first available
            idx = _PV_INDEX.get_indexer([timestamp], method='bfill')[0]
        return pv_op_data_df.iloc[idx][pv_tag] if idx >= 0 else np.nan
    except Exception:
        return np.nan


def pv_window(t_end, minutes, pv_tag):
    """Backward window [t_end - minutes, t_end] for a tag (causal). Returns a Series."""
    if pv_tag not in pv_op_data_df.columns:
        return pd.Series(dtype=float)
    t_end = _norm_ts(t_end)
    return pv_op_data_df.loc[t_end - pd.Timedelta(minutes=minutes): t_end, pv_tag].dropna()


def merge_subminute_actions(actions_in):
    """Collapse multiple actions on the same tag within the same minute into one net action.
    PrevValue = first action's prev, Value = last action's value, magnitude = net change."""
    cols = ['Source', 'VT_Start', 'Value', 'PrevValue', 'Value_num', 'PrevValue_num',
            'Description', 'num_raw_actions', 'magnitude']
    if len(actions_in) == 0:
        return pd.DataFrame(columns=cols)
    a = actions_in.copy()
    a['Value_num'] = pd.to_numeric(a['Value'], errors='coerce')
    a['PrevValue_num'] = pd.to_numeric(a['PrevValue'], errors='coerce')
    a = a.dropna(subset=['Value_num', 'PrevValue_num'])
    if len(a) == 0:
        return pd.DataFrame(columns=cols)
    a = a.drop_duplicates(subset=['VT_Start', 'Source', 'Value'])
    a['minute_floor'] = a['VT_Start'].dt.floor('min')
    a = a.sort_values(['Source', 'VT_Start'])
    rec = []
    for (source, minute), g in a.groupby(['Source', 'minute_floor']):
        g = g.sort_values('VT_Start')
        rec.append({
            'Source': source, 'VT_Start': minute,
            'Value': g.iloc[-1]['Value_num'], 'PrevValue': g.iloc[0]['PrevValue_num'],
            'Value_num': g.iloc[-1]['Value_num'], 'PrevValue_num': g.iloc[0]['PrevValue_num'],
            'Description': g['Description'].mode().iloc[0] if 'Description' in g.columns and len(g['Description'].mode()) else None,
            'num_raw_actions': len(g),
        })
    out = pd.DataFrame(rec)
    out['magnitude'] = out['Value_num'] - out['PrevValue_num']
    return out


print('Helpers defined: get_pv_at(), pv_window(), merge_subminute_actions()')


Helpers defined: get_pv_at(), pv_window(), merge_subminute_actions()


In [6]:
# ── Causal feature builder (every feature computable from data <= t; no future leakage) ──
FEATURE_NAMES = [
    'tgt_dist_to_limit',  # PV_1071(t) - 28.75   (raw PV units; <0 means in alarm)  -> severity
    'tgt_norm_pos',       # (PV_1071 - lower)/range
    'tgt_roc_15',         # (PV_1071(t) - PV_1071(t-15)) / 15   units/min
    'tgt_roc_5',          # (PV_1071(t) - PV_1071(t-5)) / 5     units/min
    'tgt_vol_30',         # std(PV_1071) over [t-30, t]
    'feed_norm_pos',      # 02FI_1000 position within its limits
    'feed_roc_15',        # 02FI_1000 normalized 15-min rate of change
    'pres_norm_pos',      # 03PIC_1013 position within its limits
    'pres_roc_15',        # 03PIC_1013 normalized 15-min rate of change
    'in_alarm',           # 1 if PV_1071(t) < 28.75 else 0
    'time_in_alarm',      # minutes since PV_1071 last crossed below 28.75 (0 if not in alarm)
]
MAX_ALARM_LOOKBACK = 240  # cap for time_in_alarm search


def _norm_pos(tag, t):
    lim = op_limits.get(tag)
    v = get_pv_at(t, tag + '.PV')
    return (v - lim['lower']) / lim['range'] if (lim and pd.notna(v)) else np.nan


def _roc(tag, t, minutes, normalize=False):
    """Per-minute rate of change over a backward window; optionally / operating range."""
    v_now = get_pv_at(t, tag + '.PV')
    v_past = get_pv_at(_norm_ts(t) - pd.Timedelta(minutes=minutes), tag + '.PV')
    if pd.isna(v_now) or pd.isna(v_past):
        return np.nan
    roc = (v_now - v_past) / minutes
    if normalize and tag in op_limits:
        roc = roc / op_limits[tag]['range']
    return roc


def _time_in_alarm(t):
    """Minutes the target PV has been continuously below the alarm limit, looking only backward."""
    w = pv_window(t, MAX_ALARM_LOOKBACK, TARGET_PV)
    if w.empty or w.iloc[-1] >= ALARM_THRESHOLD:
        return 0.0
    start_t = w.index[-1]
    for ts in reversed(w.index):
        if w.loc[ts] >= ALARM_THRESHOLD:
            break
        start_t = ts
    return (w.index[-1] - start_t).total_seconds() / 60.0


def build_runtime_context(t):
    """Causal context vector at timestamp t. Uses only data <= t."""
    t = _norm_ts(t)
    pv = get_pv_at(t, TARGET_PV)
    return {
        'tgt_dist_to_limit': (pv - ALARM_THRESHOLD) if pd.notna(pv) else np.nan,
        'tgt_norm_pos': _norm_pos(TARGET_TAG, t),
        'tgt_roc_15': _roc(TARGET_TAG, t, 15),
        'tgt_roc_5': _roc(TARGET_TAG, t, 5),
        'tgt_vol_30': pv_window(t, 30, TARGET_PV).std(),
        'feed_norm_pos': _norm_pos(FEED_TAG, t),
        'feed_roc_15': _roc(FEED_TAG, t, 15, normalize=True),
        'pres_norm_pos': _norm_pos(PRESSURE_TAG, t),
        'pres_roc_15': _roc(PRESSURE_TAG, t, 15, normalize=True),
        'in_alarm': 1.0 if (pd.notna(pv) and pv < ALARM_THRESHOLD) else 0.0,
        'time_in_alarm': _time_in_alarm(t),
    }


# ── Tag-aware features: the candidate/acted tag's OWN state (position + motion) ──
# Direction & size of an action on a tag depend on that tag's own current state.
# For events-only tags (HIC, 1034, 1009 — no PV column) these are NaN -> neutral after standardisation.
ACT_FEATURES = ['act_norm_pos', 'act_roc_15']
ALL_FEATURES = FEATURE_NAMES + ACT_FEATURES


def act_features(tag, t):
    return {'act_norm_pos': _norm_pos(tag, t), 'act_roc_15': _roc(tag, t, 15, normalize=True)}


# Smoke test on one known alarm onset
_demo_t = clusters_valid.loc[test_cluster_ids[0], 'cluster_start'] if test_cluster_ids else clusters_valid['cluster_start'].iloc[-1]
_demo = build_runtime_context(_demo_t)
print(f'Causal context at {_demo_t}:')
for kk in FEATURE_NAMES:
    print(f'  {kk:18s} = {_demo[kk]:.4f}' if pd.notna(_demo[kk]) else f'  {kk:18s} = NaN')
print(f'\nTag-aware features: {ACT_FEATURES} | total matching features: {len(ALL_FEATURES)}')


Causal context at 2025-01-03 06:47:57.193000:
  tgt_dist_to_limit  = 3.7454
  tgt_norm_pos       = -0.3842
  tgt_roc_15         = -0.0885
  tgt_roc_5          = -1.5074
  tgt_vol_30         = 2.5192
  feed_norm_pos      = -0.9685
  feed_roc_15        = 0.0751
  pres_norm_pos      = -0.5420
  pres_roc_15        = -0.0008
  in_alarm           = 0.0000
  time_in_alarm      = 0.0000

Tag-aware features: ['act_norm_pos', 'act_roc_15'] | total matching features: 13


### Build the decision-moment bank (offline, no root-cause labels needed)

For each **training** cluster, take every target-tag SP/OP action (sub-minute-merged), and at that action's timestamp build the **causal context** (data ≤ that moment). Each bank row = *(plant context now) → (action the operator took: tag, type, direction, magnitude)*. This is the entire training signal; root-cause labels are **not** used.


In [7]:
def build_bank(cluster_ids):
    """Build a decision-moment bank: one row per merged target-tag action with its causal context
    (including the acted tag's own state). No root-cause labels used."""
    rows = []
    n_raw = n_merged = 0
    for cid in tqdm(cluster_ids, desc='Bank clusters'):
        ca_raw = target_actions_df[target_actions_df['cluster_id'] == cid]
        if len(ca_raw) == 0:
            continue
        n_raw += len(ca_raw)
        merged = merge_subminute_actions(ca_raw)
        if len(merged) == 0:
            continue
        n_merged += len(merged)
        for _, act in merged.iterrows():
            ctx = build_runtime_context(act['VT_Start'])
            ctx.update(act_features(act['Source'], act['VT_Start']))   # acted tag's own state
            ctx.update({
                'cluster_id': cid,
                'action_timestamp': act['VT_Start'],
                'action_source': act['Source'],
                'action_type': act['Description'],
                'action_magnitude': act['magnitude'],
                'action_direction': 1 if act['magnitude'] > 0 else -1,
                'num_raw_actions': act['num_raw_actions'],
            })
            rows.append(ctx)
    return pd.DataFrame(rows), n_raw, n_merged


bank_raw, n_raw, n_merged = build_bank(train_cluster_ids)

bank = bank_raw[bank_raw['action_magnitude'].notna()].copy()
bank = bank.dropna(subset=['tgt_dist_to_limit', 'tgt_norm_pos'])
bank = bank.drop_duplicates(subset=['cluster_id', 'action_timestamp', 'action_source']).reset_index(drop=True)

print(f'Raw target actions      : {n_raw}')
print(f'After sub-minute merge  : {n_merged}')
print(f'Bank rows (usable)      : {len(bank)}  from {bank["cluster_id"].nunique()} clusters')
print(f'\nBank actions by tag:')
print(bank['action_source'].value_counts().to_string())
print(f'\nDirection balance: {dict(bank["action_direction"].value_counts())}')
print(f'In-alarm vs pre-alarm moments: {dict(bank["in_alarm"].value_counts())}')


Bank clusters:   0%|          | 0/305 [00:00<?, ?it/s]

Bank clusters: 100%|██████████| 305/305 [01:48<00:00,  2.81it/s]

Raw target actions      : 8290
After sub-minute merge  : 3614
Bank rows (usable)      : 3612  from 305 clusters

Bank actions by tag:
action_source
03HIC_1151    656
03HIC_3100    554
03PIC_1013    442
03LIC_1034    426
03LIC_1071    295
03HIC_1141    290
03HIC_3132    257
03LIC_1016    205
03FIC_3415    194
03PIC_3131    138
03LIC_1085     86
03TIC_1009     69

Direction balance: {-1: np.int64(1945), 1: np.int64(1667)}
In-alarm vs pre-alarm moments: {0.0: np.int64(2709), 1.0: np.int64(903)}


### Label each bank moment by its *measured effect* on `03LIC_1071.PV`

This is the process-dynamics grounding. For every action in the bank, look at what the **target PV actually did afterward** (response window `[t+lag, t+lag+W]`, with `lag=2`, `W=30` ≈ one time constant):
- `effect_pv_delta` = PV_1071(t+lag+W) − PV_1071(t)  → net move of the alarm variable
- `effect_slope_gain` = post-action slope − pre-action slope  → did the fall reverse?
- `effective` = the action was followed by genuine improvement (delta or slope turned up)

**Honest caveat:** this is associational, not perfectly causal — feed swings and concurrent actions confound a single move. Used in aggregate (and combined with context similarity), it still biases recommendations toward what *worked* rather than what was merely done, which is exactly the requirement.


In [8]:
# Response-dynamics constants (from RESULTS/response-dynamics-estimator: lag~0-2 min, tau~25 min)
RESP_LAG_MIN = 2
RESP_WINDOW_MIN = 30
EFF_DELTA_EPS = 0.2     # PV units: net rise to count as improvement
EFF_SLOPE_EPS = 0.05    # units/min: slope must turn up by at least this
FWD_MOVE_WIN = 60       # minutes: window over which we sum the tag's forward cumulative net move


def _slope(t0, t1):
    s = pv_op_data_df.loc[_norm_ts(t0):_norm_ts(t1), TARGET_PV].dropna()
    if len(s) < 3:
        return np.nan
    x = (s.index - s.index[0]).total_seconds().values / 60.0
    return float(np.polyfit(x, s.values, 1)[0])


def label_effects(df):
    deltas, sgains, pre_s, post_s = [], [], [], []
    for t in tqdm(df['action_timestamp'], desc='Effect labeling'):
        t = _norm_ts(t)
        pv0 = get_pv_at(t, TARGET_PV)
        pv1 = get_pv_at(t + pd.Timedelta(minutes=RESP_LAG_MIN + RESP_WINDOW_MIN), TARGET_PV)
        pre = _slope(t - pd.Timedelta(minutes=15), t)
        post = _slope(t + pd.Timedelta(minutes=RESP_LAG_MIN),
                      t + pd.Timedelta(minutes=RESP_LAG_MIN + RESP_WINDOW_MIN))
        deltas.append(pv1 - pv0 if (pd.notna(pv0) and pd.notna(pv1)) else np.nan)
        sgains.append(post - pre if (pd.notna(pre) and pd.notna(post)) else np.nan)
        pre_s.append(pre); post_s.append(post)
    out = df.copy()
    out['effect_pv_delta'] = deltas
    out['effect_slope_gain'] = sgains
    out['pre_slope'] = pre_s
    out['post_slope'] = post_s
    out['effective'] = ((out['effect_pv_delta'] > EFF_DELTA_EPS) |
                        (out['effect_slope_gain'] > EFF_SLOPE_EPS)).astype(int)
    return out


def add_forward_net_move(df, win_min=FWD_MOVE_WIN):
    """Per row, two severity-scaling magnitude targets:
      fwd_net_move      = signed sum of this tag's merged moves in [t, t+win] (same cluster)
      cluster_total_move= signed sum of this tag's merged moves over the WHOLE cluster
    Operators fix severe alarms with MORE moves, so the total intervention scales with severity;
    cluster_total_move is the stabler magnitude target for the severity→magnitude readout."""
    fwd = {}
    for (_cid, _tag), g in df.groupby(['cluster_id', 'action_source']):
        g = g.sort_values('action_timestamp')
        ts = g['action_timestamp'].values
        mags = g['action_magnitude'].values
        idxs = g.index.values
        for i, ix in enumerate(idxs):
            end = ts[i] + np.timedelta64(win_min, 'm')
            fwd[ix] = float(mags[(ts >= ts[i]) & (ts <= end)].sum())
    out = df.copy()
    out['fwd_net_move'] = out.index.map(fwd)
    out['cluster_total_move'] = out.groupby(['cluster_id', 'action_source'])['action_magnitude'].transform('sum')
    return out


bank = label_effects(bank)
bank = add_forward_net_move(bank)

print(f'Bank moments labeled: {len(bank)}')
print(f'Effective (PV improved afterward): {bank["effective"].mean()*100:.0f}%  '
      f'→ {100 - bank["effective"].mean()*100:.0f}% of operator actions did NOT help the target recover')
print(f'Median single |step|: {bank["action_magnitude"].abs().median():.1f}  vs  '
      f'median |cluster total move|: {bank["cluster_total_move"].abs().median():.1f}')
print('\n|cluster total move| by severity quartile (the magnitude target — should rise for OTS):')
_sev = pd.qcut(-bank['tgt_dist_to_limit'], 4, labels=['mild', 'moderate', 'high', 'severe'], duplicates='drop')
print(bank.assign(_sev=_sev).groupby('_sev', observed=True)['cluster_total_move']
      .apply(lambda s: s.abs().median()).round(1).to_string())


Effect labeling:   0%|          | 0/3612 [00:00<?, ?it/s]

Effect labeling: 100%|██████████| 3612/3612 [00:16<00:00, 213.90it/s]


Bank moments labeled: 3612
Effective (PV improved afterward): 65%  → 35% of operator actions did NOT help the target recover
Median single |step|: 2.0  vs  median |cluster total move|: 7.8

|cluster total move| by severity quartile (the magnitude target — should rise for OTS):
_sev
mild         6.0
moderate     5.6
high         8.0
severe      10.4


### Per-tag empirical process gain → the PV-raising direction

For each candidate tag we estimate the **sign of its effect on `03LIC_1071.PV`** by regressing the measured `effect_pv_delta` on the signed action step across all historical moments. `gain_sign = sign(slope)`:
- `gain_sign > 0` → increasing the tag raises PV → to recover a PVLO, **increase** it.
- `gain_sign < 0` → increasing the tag lowers PV → to recover, **decrease** it.

A **bootstrap sign-stability** (fraction of resamples agreeing on the sign) gives confidence. `pv_raising_dir` is the direction the recommender will use — *this is the process-dynamics direction, derived from the alarm variable's own response, not from operator votes.* We also show the operator's own majority direction so divergences are visible.


In [9]:
def estimate_tag_gain(sub, n_boot=400, seed=0):
    """Empirical gain sign of a tag on PV_1071 via regression of effect_pv_delta on signed step."""
    d = sub.dropna(subset=['action_magnitude', 'effect_pv_delta'])
    d = d[np.isfinite(d['action_magnitude']) & np.isfinite(d['effect_pv_delta'])]
    n = len(d)
    if n < 8 or d['action_magnitude'].std() < 1e-9:
        return dict(n=n, gain_slope=np.nan, gain_sign=0, confidence=0.0)
    x = d['action_magnitude'].values.astype(float)
    y = d['effect_pv_delta'].values.astype(float)
    slope = np.polyfit(x, y, 1)[0]
    rng = np.random.default_rng(seed)
    signs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if np.std(x[idx]) < 1e-9:
            continue
        signs.append(np.sign(np.polyfit(x[idx], y[idx], 1)[0]))
    signs = np.array(signs)
    maj = np.sign(slope) if slope != 0 else 1
    conf = float(np.mean(signs == maj)) if len(signs) else 0.0
    return dict(n=n, gain_slope=float(slope), gain_sign=int(np.sign(slope)) or 1, confidence=conf)


maj_dir = bank.groupby('action_source')['action_direction'].agg(lambda s: 1 if s.mean() > 0 else -1).to_dict()

gain_rows = []
for tag in ACTION_TAGS:
    sub = bank[bank['action_source'] == tag]
    g = estimate_tag_gain(sub)
    g['tag'] = tag
    g['pv_raising_dir'] = g['gain_sign']               # to raise PV move the tag this way
    g['operator_majority_dir'] = maj_dir.get(tag, np.nan)
    g['eff_rate'] = sub['effective'].mean() if len(sub) else np.nan
    gain_rows.append(g)

tag_gain = pd.DataFrame(gain_rows).set_index('tag')[
    ['n', 'gain_slope', 'gain_sign', 'confidence', 'pv_raising_dir', 'operator_majority_dir', 'eff_rate']]
PV_RAISING_DIR = tag_gain['pv_raising_dir'].to_dict()
GAIN_CONF = tag_gain['confidence'].to_dict()

pd.set_option('display.float_format', lambda v: f'{v:.3f}')
print('Per-tag empirical process gain (effect on 03LIC_1071.PV):\n')
print(tag_gain.assign(
    dir_to_raise_PV=lambda d: d['pv_raising_dir'].map({1: 'INCREASE', -1: 'DECREASE', 0: '—'}),
    agrees_with_operator=lambda d: np.where(d['pv_raising_dir'] == d['operator_majority_dir'], 'yes', 'NO')
)[['n', 'gain_slope', 'confidence', 'dir_to_raise_PV', 'agrees_with_operator', 'eff_rate']].to_string())
print('\nNote: where agrees_with_operator == NO, operators on average moved the tag the way that '
      'pushed PV further DOWN — i.e. mimicking them would worsen the alarm.')


Per-tag empirical process gain (effect on 03LIC_1071.PV):

              n  gain_slope  confidence dir_to_raise_PV agrees_with_operator  eff_rate
tag                                                                                   
03PIC_1013  442      -3.055       1.000        DECREASE                  yes     0.690
03HIC_1141  290      -0.494       0.880        DECREASE                  yes     0.652
03HIC_1151  656       0.623       0.950        INCREASE                   NO     0.649
03LIC_1071  294       0.580       1.000        INCREASE                  yes     0.678
03HIC_3132  257       0.268       1.000        INCREASE                  yes     0.630
03HIC_3100  554      -0.040       0.655        DECREASE                  yes     0.655
03FIC_3415  194      -0.054       0.745        DECREASE                  yes     0.624
03LIC_1034  426       0.135       0.905        INCREASE                  yes     0.631
03PIC_3131  138       0.134       0.863        INCREASE                

In [10]:
# Robust standardisation (median / IQR) fit on the bank — the validated recipe. Each feature is
# centred & scaled independently, so mixing units (PV, units/min, fractions, minutes, binary) is fine.
def fit_scaler(df, feature_names):
    med = df[feature_names].median()
    iqr = df[feature_names].quantile(0.75) - df[feature_names].quantile(0.25)
    std = df[feature_names].std()
    scale = iqr.copy()
    scale[scale <= 1e-9] = std[scale <= 1e-9]   # fall back to std where IQR degenerate
    scale[scale <= 1e-9] = 1.0                   # then to 1.0 (constant feature)
    return {'median': med, 'scale': scale}


def standardize(df_or_dict, scaler, feature_names):
    """Return standardized array (NaN→0 after centring, so a missing feature is 'neutral')."""
    if isinstance(df_or_dict, dict):
        vals = np.array([df_or_dict.get(f, np.nan) for f in feature_names], dtype=float)
        z = (vals - scaler['median'][feature_names].values) / scaler['scale'][feature_names].values
        return np.nan_to_num(z, nan=0.0)
    z = (df_or_dict[feature_names] - scaler['median']) / scaler['scale']
    return np.nan_to_num(z.values, nan=0.0)


scaler = fit_scaler(bank, ALL_FEATURES)
bank_Z = standardize(bank, scaler, ALL_FEATURES)   # (n_bank, n_features) standardized matrix

print(f'Robust scaler fit on bank over {len(ALL_FEATURES)} features.')
print(f'Standardized bank matrix: {bank_Z.shape}')


Robust scaler fit on bank over 13 features.
Standardized bank matrix: (3612, 13)


In [19]:
# Persist bank (with effect labels), scaler, gain table + metadata for reloadable runtime use
output_dir = '/home/h604827/ControlActions/RESULTS/similarity_test_results/runtime_recommender_v2'
os.makedirs(output_dir, exist_ok=True)

bank.to_csv(f'{output_dir}/decision_moment_bank.csv', index=False)
tag_gain.to_csv(f'{output_dir}/tag_process_gain.csv')
with open(f'{output_dir}/scaler.json', 'w') as f:
    json.dump({'feature_names': ALL_FEATURES,
               'median': scaler['median'].to_dict(),
               'scale': scaler['scale'].to_dict()}, f, indent=2)
with open(f'{output_dir}/metadata.json', 'w') as f:
    json.dump({'pv_op_path': PV_OP_PATH,
               'train_cluster_ids': train_cluster_ids,
               'test_cluster_ids': test_cluster_ids,
               'trip_clusters_excluded': sorted(trip_clusters),
               'alarm_threshold': ALARM_THRESHOLD,
               'action_tags': ACTION_TAGS,
               'resp_lag_min': RESP_LAG_MIN, 'resp_window_min': RESP_WINDOW_MIN,
               'bank_rows': len(bank),
               'pv_raising_dir': PV_RAISING_DIR}, f, indent=2, default=str)

print(f'Saved to {output_dir}/')
print('  decision_moment_bank.csv, tag_process_gain.csv, scaler.json, metadata.json')


Saved to /home/h604827/ControlActions/RESULTS/similarity_test_results/runtime_recommender_v2/
  decision_moment_bank.csv, tag_process_gain.csv, scaler.json, metadata.json


## Equilibrium Cascade Controller (primary recommender)

Below: (1) estimate the cascade **process-gain sensitivities** from data, (2) define `recommend_cascade(snapshot)` which sizes the coordinated 1071 + 1016 OP moves to restore equilibrium. The recommender takes a **live snapshot** of `{pv/op of 1071 and 1016}` — exactly what the OTS harness can supply — so it is deployment-ready.


In [26]:
# ── Cascade structure + process-gain sensitivities (estimated from OP CHANGE events) ──
CASCADE = {
    'primary':  {'tag': '03LIC_1071', 'pv': '03LIC_1071.PV', 'op': '03LIC_1071.OP'},
    'supplier': {'tag': '03LIC_1016', 'pv': '03LIC_1016.PV', 'op': '03LIC_1016.OP'},
}


def estimate_op_pv_sensitivity(op_source, pv_tag, lag=2, win=30, lo=1, hi=15):
    """Median PV change over `win` min per unit OP step, measured from that tag's OP CHANGE events.
    Captures the sign + size of the process gain (response-dynamics: lag~0-2 min, tau~25 min)."""
    e = actions_df[(actions_df['Source'] == op_source) & (actions_df['Description'] == 'OP')].copy()
    e['v'] = pd.to_numeric(e['Value'], errors='coerce')
    e['p'] = pd.to_numeric(e['PrevValue'], errors='coerce')
    e['dop'] = e['v'] - e['p']
    e = e[e['dop'].abs().between(lo, hi)].dropna(subset=['dop'])
    rates = []
    for _, r in e.iterrows():
        t = _norm_ts(r['VT_Start'])
        pv0 = get_pv_at(t + pd.Timedelta(minutes=lag), pv_tag)
        pv1 = get_pv_at(t + pd.Timedelta(minutes=lag + win), pv_tag)
        if pd.notna(pv0) and pd.notna(pv1):
            rates.append((pv1 - pv0) / r['dop'])
    rates = np.array([x for x in rates if np.isfinite(x)])
    return float(np.median(rates)) if len(rates) else np.nan


SENS_1071 = estimate_op_pv_sensitivity('03LIC_1071', '03LIC_1071.PV')          # +PV per +%OP
SENS_1016 = estimate_op_pv_sensitivity('03LIC_1016', '03LIC_1016.PV')
COUPLING_1071OP_1016PV = estimate_op_pv_sensitivity('03LIC_1071', '03LIC_1016.PV')  # confounded sign

print('Cascade process-gain sensitivities (PV units per unit OP, 30-min response):')
print(f'  1071.OP → 1071.PV : {SENS_1071:+.2f}   (+ : opening inflow raises 1071 level)')
print(f'  1016.OP → 1016.PV : {SENS_1016:+.2f}   (+ : opening inflow refills 1016)')
print(f'  1071.OP → 1016.PV : {COUPLING_1071OP_1016PV:+.2f}   (event-based estimate is confounded by')
print('                       closed-loop control → sign unreliable; supplier protection therefore')
print('                       uses the REACTIVE trigger: act on 1016 when its level ≤ floor.)')

# ── Recovery targets / guards ──
TARGET_1071 = op_limits['03LIC_1071']['lower']   # 35.25 — recover the level above the 28.75 alarm with margin
FLOOR_1016  = op_limits['03LIC_1016']['lower']   # 38.00 — keep the supplier above its floor
MAX_OP_STEP = 15.0                                # safety cap on a single recommended OP move
OP_HI = 100.0                                     # physical OP clamp


def make_snapshot(t):
    """Live snapshot of cascade PV/OP at timestamp t (the OTS harness supplies these directly)."""
    t = _norm_ts(t)
    return {'pv_1071': get_pv_at(t, '03LIC_1071.PV'), 'op_1071': get_pv_at(t, '03LIC_1071.OP'),
            'pv_1016': get_pv_at(t, '03LIC_1016.PV'), 'op_1016': get_pv_at(t, '03LIC_1016.OP')}


def recommend_cascade(snapshot, target_1071=TARGET_1071, floor_1016=FLOOR_1016,
                      margin=1.0, max_step=MAX_OP_STEP):
    """Equilibrium cascade recommendation for a 1071 PVLO.
      1) increase 1071 OP to lift the level back to target  (size = deficit / gain_1071)
      2) if the supplier 1016 is / will be below its floor, increase 1016 OP to refill it
    Direction is fixed by the physics (increase); magnitude scales with the deficits → intensity."""
    pv1071, op1071 = snapshot['pv_1071'], snapshot['op_1071']
    pv1016, op1016 = snapshot['pv_1016'], snapshot['op_1016']

    # primary: lift 1071
    deficit_1071 = max(0.0, (target_1071 + margin) - pv1071)
    d_op1071 = min(max_step, deficit_1071 / SENS_1071) if deficit_1071 > 0 else 0.0
    d_op1071 = min(d_op1071, OP_HI - op1071)

    # cascade: protect supplier 1016 (account for the extra drop our 1071 pull will cause)
    pred_drop = max(0.0, -COUPLING_1071OP_1016PV) * d_op1071
    pv1016_eff = pv1016 - pred_drop
    deficit_1016 = max(0.0, (floor_1016 + margin) - pv1016_eff)
    d_op1016 = min(max_step, deficit_1016 / SENS_1016) if deficit_1016 > 0 else 0.0
    d_op1016 = min(d_op1016, OP_HI - op1016)

    return [
        {'tag': '03LIC_1071', 'signal': 'OP', 'direction': 'increase' if d_op1071 > 0 else 'hold',
         'delta_op': round(d_op1071, 2), 'from_op': round(op1071, 1), 'to_op': round(op1071 + d_op1071, 1),
         'pv_now': round(pv1071, 1), 'deficit': round(deficit_1071, 1),
         'reason': f'raise 1071 level toward {target_1071 + margin:.1f} (gain {SENS_1071:.2f} PV/%OP)'},
        {'tag': '03LIC_1016', 'signal': 'OP', 'direction': 'increase' if d_op1016 > 0 else 'hold',
         'delta_op': round(d_op1016, 2), 'from_op': round(op1016, 1), 'to_op': round(op1016 + d_op1016, 1),
         'pv_now': round(pv1016, 1), 'pv_after_pull': round(pv1016_eff, 1), 'deficit': round(deficit_1016, 1),
         'reason': (f'refill 1016 toward {floor_1016 + margin:.1f}; reaches {pv1016_eff:.1f} after 1071 pull'
                    if d_op1016 > 0 else 'supplier above floor — hold')},
    ]


# Demo at a real severe 2025 alarm moment
_al = pv_op_data_df.loc['2025-01-01':]
_al = _al[(_al['03LIC_1071.PV'] < ALARM_THRESHOLD) & (_al['03LIC_1071.PV'] > 0)]
_t = _al['03LIC_1071.PV'].idxmin()
print(f'\nExample — real alarm at {_t}:')
_snap = make_snapshot(_t)
print('  snapshot:', {k: round(v, 1) for k, v in _snap.items()})
for a in recommend_cascade(_snap):
    print(f"  → {a['tag']}.{a['signal']}: {a['direction']} by {a['delta_op']}  "
          f"({a['from_op']}→{a['to_op']})  | {a['reason']}")


Cascade process-gain sensitivities (PV units per unit OP, 30-min response):
  1071.OP → 1071.PV : +2.37   (+ : opening inflow raises 1071 level)
  1016.OP → 1016.PV : +3.70   (+ : opening inflow refills 1016)
  1071.OP → 1016.PV : +2.58   (event-based estimate is confounded by
                       closed-loop control → sign unreliable; supplier protection therefore
                       uses the REACTIVE trigger: act on 1016 when its level ≤ floor.)

Example — real alarm at 2025-01-12 09:50:00:
  snapshot: {'pv_1071': np.float64(0.1), 'op_1071': np.float64(58.0), 'pv_1016': np.float64(42.8), 'op_1016': np.float64(51.1)}
  → 03LIC_1071.OP: increase by 15.0  (58.0→73.0)  | raise 1071 level toward 36.2 (gain 2.37 PV/%OP)
  → 03LIC_1016.OP: hold by 0.0  (51.1→51.1)  | supplier above floor — hold


### Validation on real 2025 alarms — does the cascade scale with intensity & make sense?

Sample real 2025 1071-PVLO moments in three depth bands (mild / moderate / severe), run `recommend_cascade` on each, and check:
- **Direction** is always *increase 1071 OP* (physics-fixed). ✓ by construction
- **Magnitude** of the 1071 OP step **grows with alarm depth** (the intensity-scaling the OTS wants).
- **Supplier protection** kicks in more often / larger as severity grows (1016 increasingly depleted).
- **Sanity vs operators:** compare our recommended 1071-OP increase to the OP increase operators actually applied in those moments (cross-check only — operators aren't ground truth).


In [27]:
# Validation on REAL 2025 alarm moments across three intensity bands
d25 = pv_op_data_df.loc['2025-01-01':]
depth25 = ALARM_THRESHOLD - d25['03LIC_1071.PV']
DEPTH_BANDS = {'mild (0–2)': (0, 2), 'moderate (2–6)': (2, 6), 'severe (>6)': (6, 40)}
rng_v = np.random.default_rng(1)

rows = []
for name, (lo, hi) in DEPTH_BANDS.items():
    idx = depth25[(depth25 >= lo) & (depth25 < hi)].index
    idx = idx[d25.loc[idx, '03LIC_1071.PV'] > 0]            # exclude trips
    if len(idx) == 0:
        continue
    pick = rng_v.choice(len(idx), size=min(80, len(idx)), replace=False)
    for t in idx[pick]:
        snap = make_snapshot(t)
        if any(pd.isna(v) for v in snap.values()):
            continue
        a71, a16 = recommend_cascade(snap)
        # what the operator actually did to 1071 OP over the next 30 min (sanity cross-check)
        op_future = pv_op_data_df.loc[t: t + pd.Timedelta(minutes=30), '03LIC_1071.OP'].dropna()
        op_move = (op_future.max() - snap['op_1071']) if len(op_future) else np.nan
        rows.append({'band': name, 'depth': round(ALARM_THRESHOLD - snap['pv_1071'], 1),
                     'rec_dOP_1071': a71['delta_op'], 'rec_dir_1071': a71['direction'],
                     'rec_dOP_1016': a16['delta_op'], 'acts_1016': int(a16['delta_op'] > 0),
                     'oper_dOP_1071': round(op_move, 1) if pd.notna(op_move) else np.nan})
val = pd.DataFrame(rows)

summary = val.groupby('band', sort=False).agg(
    n=('depth', 'size'), med_depth=('depth', 'median'),
    med_rec_dOP_1071=('rec_dOP_1071', 'median'),
    pct_increase_1071=('rec_dir_1071', lambda s: round(100 * (s == 'increase').mean())),
    pct_1016_action=('acts_1016', lambda s: round(100 * s.mean())),
    med_rec_dOP_1016=('rec_dOP_1016', 'median'),
    med_oper_dOP_1071=('oper_dOP_1071', 'median'))
print('Cascade recommendation on real 2025 alarms, by intensity band:\n')
print(summary.to_string())
print('\n• Direction: 1071 OP increase in ~100% of alarm moments (physics-fixed).')
print('• Magnitude (med_rec_dOP_1071) GROWS mild → severe = intensity scaling.')
print('• Supplier (1016) action fires more & larger as severity grows.')
print('• med_oper_dOP_1071 = what operators actually added to 1071 OP (same-sign sanity cross-check).')

val.to_csv(f'{output_dir}/cascade_validation_2025.csv', index=False)
print(f'\nSaved -> {output_dir}/cascade_validation_2025.csv')


Cascade recommendation on real 2025 alarms, by intensity band:

                 n  med_depth  med_rec_dOP_1071  pct_increase_1071  pct_1016_action  med_rec_dOP_1016  med_oper_dOP_1071
band                                                                                                                    
mild (0–2)      80      0.900             3.545                100               24             0.000              1.350
moderate (2–6)  80      3.500             4.650                100               44             0.000              2.300
severe (>6)     80     16.500            10.120                100               34             0.000              0.000

• Direction: 1071 OP increase in ~100% of alarm moments (physics-fixed).
• Magnitude (med_rec_dOP_1071) GROWS mild → severe = intensity scaling.
• Supplier (1016) action fires more & larger as severity grows.
• med_oper_dOP_1071 = what operators actually added to 1071 OP (same-sign sanity cross-check).

Saved -> /home/h604827/Co

## OTS demo — three feed-variation scenarios of increasing intensity

The OTS engineers will inject a feed disturbance at three magnitudes → three 1071 PVLO alarms of increasing intensity. We represent each by a realistic plant **snapshot** (deeper 1071 level + more 1016 depletion as the feed cut grows) and run the cascade controller. Expected, and shown below: **direction stays "increase 1071 OP", magnitude grows S1→S3, and the supplier-protection action on 1016 engages as 1016 falls toward its floor.**


In [28]:
# ── Three OTS feed-variation scenarios (increasing intensity) ──
# Snapshots use realistic in-alarm values: 1071 deeper below limit and 1016 more depleted as the
# feed cut grows (data during alarm: 1071.PV~24, 1016.PV~37, 1071.OP~57, 1016.OP~44).
OTS_SCENARIOS = {
    'S1 mild feed cut':     {'pv_1071': 28.0, 'op_1071': 54.0, 'pv_1016': 40.0, 'op_1016': 38.0},
    'S2 moderate feed cut': {'pv_1071': 23.0, 'op_1071': 56.0, 'pv_1016': 37.0, 'op_1016': 41.0},
    'S3 severe feed cut':   {'pv_1071': 15.0, 'op_1071': 58.0, 'pv_1016': 34.0, 'op_1016': 44.0},
}

print('Equilibrium cascade recommendations across the three OTS scenarios:\n')
demo_rows = []
for name, snap in OTS_SCENARIOS.items():
    acts = recommend_cascade(snap)
    print(f'── {name}  (1071.PV={snap["pv_1071"]}, 1016.PV={snap["pv_1016"]}) ──')
    for a in acts:
        print(f'    {a["tag"]}.{a["signal"]:2s}: {a["direction"]:8s} by {a["delta_op"]:5.2f}  '
              f'({a["from_op"]}→{a["to_op"]})  | {a["reason"]}')
    demo_rows.append({'scenario': name, 'depth_1071': round(ALARM_THRESHOLD - snap['pv_1071'], 1),
                      'dOP_1071': acts[0]['delta_op'], 'dOP_1016': acts[1]['delta_op']})
    print()

demo = pd.DataFrame(demo_rows)
print('Summary (recommended OP increases grow with feed-cut intensity):')
print(demo.to_string(index=False))

fig = go.Figure()
fig.add_trace(go.Bar(x=demo['scenario'], y=demo['dOP_1071'], name='03LIC_1071.OP ↑ (lift level)',
                     marker_color='steelblue', text=demo['dOP_1071'], textposition='outside'))
fig.add_trace(go.Bar(x=demo['scenario'], y=demo['dOP_1016'], name='03LIC_1016.OP ↑ (protect supplier)',
                     marker_color='indianred', text=demo['dOP_1016'], textposition='outside'))
fig.update_layout(barmode='group', height=460, width=820,
                  title='OTS scenarios: recommended OP increases scale with feed-cut intensity',
                  xaxis_title='scenario (mild → severe)', yaxis_title='recommended OP increase (%)')
fig.show()
fig.write_html(f'{output_dir}/ots_cascade_scenarios.html')
print(f'\nSaved -> {output_dir}/ots_cascade_scenarios.html')


Equilibrium cascade recommendations across the three OTS scenarios:

── S1 mild feed cut  (1071.PV=28.0, 1016.PV=40.0) ──
    03LIC_1071.OP: increase by  3.48  (54.0→57.5)  | raise 1071 level toward 36.2 (gain 2.37 PV/%OP)
    03LIC_1016.OP: hold     by  0.00  (38.0→38.0)  | supplier above floor — hold

── S2 moderate feed cut  (1071.PV=23.0, 1016.PV=37.0) ──
    03LIC_1071.OP: increase by  5.59  (56.0→61.6)  | raise 1071 level toward 36.2 (gain 2.37 PV/%OP)
    03LIC_1016.OP: increase by  0.54  (41.0→41.5)  | refill 1016 toward 39.0; reaches 37.0 after 1071 pull

── S3 severe feed cut  (1071.PV=15.0, 1016.PV=34.0) ──
    03LIC_1071.OP: increase by  8.97  (58.0→67.0)  | raise 1071 level toward 36.2 (gain 2.37 PV/%OP)
    03LIC_1016.OP: increase by  1.35  (44.0→45.4)  | refill 1016 toward 39.0; reaches 34.0 after 1071 pull

Summary (recommended OP increases grow with feed-cut intensity):
            scenario  depth_1071  dOP_1071  dOP_1016
    S1 mild feed cut       0.800     3.480     


Saved -> /home/h604827/ControlActions/RESULTS/similarity_test_results/runtime_recommender_v2/ots_cascade_scenarios.html


## Similarity retrieval (revived) — context-matched historical actions

The cascade controller above ignores the bank. Here we revive the original idea: at a moment, **compare the plant context to every historical action and pull the nearest ones**, then aggregate their direction + magnitude. This reuses `bank_Z` (standardised context), `scaler`, and `PV_RAISING_DIR`. Magnitude uses `cluster_total_move`; we restrict neighbours to those moves that were **effective**. We then measure how it performs on held-out 2025 alarms and compare to the cascade controller.


In [11]:
# ── k-NN context retrieval over the bank ──
_idx_by_tag = {t: np.where(bank['action_source'].values == t)[0] for t in ACTION_TAGS}
_eff = bank['effective'].values
_dir = bank['action_direction'].values
_ctm = bank['cluster_total_move'].values
_typ = bank['action_type'].values


def _wmed(v, w):
    v = np.asarray(v, float); w = np.asarray(w, float); o = np.argsort(v); v, w = v[o], w[o]
    cw = np.cumsum(w); return float(v[np.searchsorted(cw, cw[-1] / 2.0)]) if cw[-1] > 0 else float(np.median(v))


def retrieve(t_now, tag, k=15, effective_only=True):
    """Return the k most context-similar historical actions on `tag`, + aggregated recommendation."""
    idx = _idx_by_tag.get(tag, np.array([], int))
    if effective_only:
        idx = idx[_eff[idx] == 1]
    if len(idx) == 0:
        return None
    zq = standardize({**build_runtime_context(t_now), **act_features(tag, t_now)}, scaler, ALL_FEATURES)
    d = np.sqrt(((bank_Z[idx] - zq) ** 2).sum(1))
    sel = idx[np.argsort(d)[:k]]; dd = np.sort(d)[:k]; w = np.exp(-(dd / (np.median(dd) + 1e-9)) ** 2)
    vote = np.sum(w * _dir[sel]) / w.sum()
    nbrs = bank.iloc[sel][['cluster_id', 'action_timestamp', 'action_direction', 'cluster_total_move', 'effective']].copy()
    nbrs['dist'] = dd.round(2)
    return {'tag': tag, 'direction': 1 if vote > 0 else -1, 'dir_conf': abs(vote),
            'magnitude': (1 if vote > 0 else -1) * abs(_wmed(np.abs(_ctm[sel]), w)),
            'similarity': 1 / (1 + dd.mean()), 'support': len(sel), 'neighbours': nbrs}


# Example: retrieve neighbours for a real severe alarm, for the two cascade tags
_al = pv_op_data_df.loc['2025-01-01':]
_t = _al[(_al['03LIC_1071.PV'] < ALARM_THRESHOLD) & (_al['03LIC_1071.PV'] > 0)]['03LIC_1071.PV'].idxmin()
print(f'Retrieval at real alarm {_t}:')
for tag in ['03LIC_1071', '03LIC_1016', '03PIC_1013']:
    r = retrieve(_t, tag, k=10)
    if r:
        print(f"  {tag}: dir={'inc' if r['direction']>0 else 'dec'} (conf {r['dir_conf']:.2f}) "
              f"mag~{r['magnitude']:+.1f}  sim={r['similarity']:.2f}  vs gain_dir={PV_RAISING_DIR[tag]:+d}")


Retrieval at real alarm 2025-01-12 09:50:00:
  03LIC_1071: dir=dec (conf 0.38) mag~-0.2  sim=0.34  vs gain_dir=+1
  03LIC_1016: dir=inc (conf 0.14) mag~+2.0  sim=0.23  vs gain_dir=+1
  03PIC_1013: dir=dec (conf 0.53) mag~-6.0  sim=0.29  vs gain_dir=-1


In [12]:
# ── How well does context retrieval perform on held-out 2025 alarms? ──
ev_rows = []
for cid in test_cluster_ids:
    m = merge_subminute_actions(target_actions_df[target_actions_df['cluster_id'] == cid])
    for _, a in m.iterrows():
        ev_rows.append({'t': a['VT_Start'], 'tag': a['Source']})
ev = pd.DataFrame(ev_rows)

rec = []
for _, r in tqdm(ev.iterrows(), total=len(ev), desc='Retrieval eval'):
    out = retrieve(r['t'], r['tag'], k=15)
    if out is None:
        continue
    depth = ALARM_THRESHOLD - get_pv_at(r['t'], TARGET_PV)
    rec.append({'tag': r['tag'], 'depth': depth, 'sim': out['similarity'], 'support': out['support'],
                'dir': out['direction'], 'gain_dir': PV_RAISING_DIR.get(r['tag'], 0),
                'abs_mag': abs(out['magnitude']), 'nbr_eff': out['neighbours']['effective'].mean()})
R = pd.DataFrame(rec)
R['dir_eq_gain'] = R['dir'] == R['gain_dir']

print(f'Held-out moments scored: {len(R)}')
print(f'Retrieved direction == physics gain direction : {R["dir_eq_gain"].mean()*100:.0f}%')
print(f'Mean neighbour effectiveness (retrieved are "what worked"): {R["nbr_eff"].mean()*100:.0f}%')
print(f'Mean similarity score: {R["sim"].mean():.2f}  | mean support: {R["support"].mean():.0f}')
print('\nMagnitude vs alarm depth (retrieval scales with severity?):')
R['band'] = pd.cut(R['depth'], [-1, 2, 6, 50], labels=['mild', 'moderate', 'severe'])
print(R.groupby('band', observed=True)['abs_mag'].median().round(1).to_string())
print('\nDirection-vs-gain agreement by tag:')
print((R.groupby('tag')['dir_eq_gain'].mean()*100).round(0).sort_values(ascending=False).to_string())


Retrieval eval: 100%|██████████| 723/723 [00:50<00:00, 14.22it/s]

Held-out moments scored: 723
Retrieved direction == physics gain direction : 58%
Mean neighbour effectiveness (retrieved are "what worked"): 100%
Mean similarity score: 0.32  | mean support: 15

Magnitude vs alarm depth (retrieval scales with severity?):
band
mild       10.000
moderate   10.000
severe      8.000

Direction-vs-gain agreement by tag:
tag
03FIC_3415   100.000
03HIC_1141   100.000
03TIC_1009   100.000
03HIC_3132    94.000
03LIC_1071    92.000
03HIC_3100    90.000
03LIC_1016    80.000
03LIC_1034    47.000
03PIC_1013    46.000
03LIC_1085    23.000
03HIC_1151    18.000
03PIC_3131    14.000


In [13]:
# ── Is "PV up after 30 min" a valid success label? noise + mean-reversion check ──
pv = pv_op_data_df[TARGET_PV].dropna()
# 1) raw fluctuation: how much does PV wander over 32 min, vs our 0.2 "improvement" threshold
roll = pv.rolling(32).std().dropna()
print(f'PV 32-min rolling std: median={roll.median():.2f}, p90={roll.quantile(.9):.2f}  (EFF_DELTA_EPS=0.2)')

# 2) single endpoint vs robust window: how often does the sign flip?
acts = bank['action_timestamp'].map(_norm_ts)
pt = bank['effect_pv_delta'].values
win = []
for t in acts:
    pre = pv.loc[t - pd.Timedelta(minutes=15):t]; post = pv.loc[t + pd.Timedelta(minutes=2):t + pd.Timedelta(minutes=32)]
    win.append(post.mean() - pre.mean() if len(pre) and len(post) else np.nan)
win = np.array(win); ok = np.isfinite(pt) & np.isfinite(win)
print(f'sign(endpoint Δ) != sign(window Δ): {(np.sign(pt[ok])!=np.sign(win[ok])).mean()*100:.0f}% of actions disagree')

# 3) mean-reversion baseline: low-PV moments with NO action ±60 min — does PV rise anyway?
act_t = pd.DatetimeIndex(sorted(acts)); below = pv[pv < pv.quantile(.15)]
rng2 = np.random.default_rng(0); base = []
for t in below.sample(min(2000, len(below)), random_state=0).index:
    if act_t[np.searchsorted(act_t, t).clip(0, len(act_t)-1)] - t < pd.Timedelta(minutes=60) and \
       t - act_t[(np.searchsorted(act_t, t)-1).clip(0)] < pd.Timedelta(minutes=60): continue
    p1 = pv.loc[t + pd.Timedelta(minutes=32):t + pd.Timedelta(minutes=37)]
    if len(p1): base.append(p1.mean() - pv.loc[t])
base = np.array(base)
print(f'\nNO-action low-PV baseline: {(base>0.2).mean()*100:.0f}% "recover" in 30min anyway (mean Δ {base.mean():+.2f})')
print(f'Post-action effective rate: {bank["effective"].mean()*100:.0f}%  → action lift over reversion ≈ {bank["effective"].mean()*100-(base>0.2).mean()*100:+.0f}pts')


PV 32-min rolling std: median=1.10, p90=3.17  (EFF_DELTA_EPS=0.2)
sign(endpoint Δ) != sign(window Δ): 40% of actions disagree

NO-action low-PV baseline: 71% "recover" in 30min anyway (mean Δ +1.32)
Post-action effective rate: 65%  → action lift over reversion ≈ -6pts


## Phase-wise view — the plant is not one regime

Global stats average over years of campaigns, trips and feed changes. Here we profile `03LIC_1071.PV` **per month**: typical level (median + IQR), 32-min volatility, % of time below the alarm, and the no-action reversion rate. If level/volatility/reversion drift across phases, no single global gain or success-threshold is valid.


In [14]:
# ── Monthly phase profile of the alarm tag ──
pvm = pv.copy(); pvm.index = pd.to_datetime(pvm.index)
mo = pvm.resample('MS')
prof = pd.DataFrame({
    'days': mo.apply(lambda s: s.index.normalize().nunique()),
    'pv_med': mo.median(), 'pv_p25': mo.quantile(.25), 'pv_p75': mo.quantile(.75),
    'vol32': pvm.rolling(32).std().resample('MS').median(),
    'pct_below': mo.apply(lambda s: (s < ALARM_THRESHOLD).mean()*100),
}).dropna(subset=['pv_med'])
prof['iqr'] = (prof['pv_p75'] - prof['pv_p25']).round(1)
prof = prof.round(1)
print('Monthly phase profile (median level shifts → no single steady-state):')
print(prof[['days','pv_med','iqr','vol32','pct_below']].to_string())
print(f"\nLevel range across months: median PV {prof['pv_med'].min():.0f}–{prof['pv_med'].max():.0f} "
      f"(alarm at {ALARM_THRESHOLD}); IQR {prof['iqr'].min()}–{prof['iqr'].max()}; "
      f"%below {prof['pct_below'].min():.0f}–{prof['pct_below'].max():.0f}%")

# Reversion rate per half-year phase — does the baseline bounce hold everywhere?
print('\nNo-action low-PV reversion rate by phase:')
for ph, s in pvm.groupby(pvm.index.to_period('6M')):
    thr = s.quantile(.15); lo = s[s < thr]
    if len(lo) < 100: continue
    d = [pvm.loc[t+pd.Timedelta(minutes=32):t+pd.Timedelta(minutes=37)].mean()-pvm.loc[t]
         for t in lo.sample(min(800,len(lo)),random_state=0).index]
    d = np.array([x for x in d if np.isfinite(x)])
    print(f'  {ph}: PVmed {s.median():4.0f} | {(d>0.2).mean()*100:.0f}% recover w/o action (Δ {d.mean():+.1f})')


Monthly phase profile (median level shifts → no single steady-state):
            days  pv_med    iqr  vol32  pct_below
TimeStamp                                        
2022-01-01    29  37.100  2.300  1.500      0.500
2022-02-01    28  35.400  2.800  0.900      0.500
2022-03-01    30  38.700  1.600  0.600      0.100
2022-04-01    30  36.000  2.400  0.500      0.300
2022-05-01    31  36.000  0.800  0.500      0.100
2022-06-01    30  35.100  1.800  0.500      0.800
2022-07-01    31  35.000  0.700  0.400      0.100
2022-08-01    31  43.900  1.200  0.500      0.000
2022-09-01    30  44.000  0.900  0.500      0.000
2022-10-01    29  33.600  3.400  0.500      2.400
2022-11-01    30  37.400  4.400  0.600      0.300
2022-12-01    30  35.700  4.700  0.600      0.200
2023-01-01    31  41.900  1.500  0.800      0.200
2023-02-01    28  35.900  1.600  0.700      0.100
2023-03-01    30  35.100  1.200  0.700      0.200
2023-04-01    30  35.000  1.300  0.800      0.600
2023-05-01    31  35.000  1.30